# DeepSpeed ZeRO: Zero Redundancy Optimizer

> ⚠️ **Hardware Note**: This notebook runs on a single GPU (T4/A10/A100).
> Multi-GPU ZeRO benefits are illustrated with memory math; the actual training loop
> runs on 1 GPU using `WORLD_SIZE=1`, which exercises the ZeRO-1 code path.

## Historical Context

**ZeRO** (Zero Redundancy Optimizer) was published by Microsoft in 2020
(Rajbhandari et al., NeurIPS 2020). It addressed a fundamental inefficiency in
data-parallel training: every GPU holds an **identical copy** of optimizer states,
gradients, and parameters. With 8 GPUs, 7/8 of that memory is pure redundancy.

## What Each Stage Shards

| Stage | What is partitioned | Memory per GPU (16P total) |
|-------|---------------------|---------------------------|
| Baseline (DDP) | Nothing | 16P |
| ZeRO-1 | Optimizer states | P × (6 + 2 + 8/N) |
| ZeRO-2 | + Gradients | P × (6 + 10/N) |
| ZeRO-3 | + Parameters | 16P / N |

Where P = parameters, N = number of GPUs.

With 8 GPUs:
- ZeRO-1: saves 28 GB (optimizer) per GPU on a 7B model
- ZeRO-2: saves 28 + 14 GB per GPU
- ZeRO-3: model memory scales linearly with N — 8× reduction

In [ ]:
# ── Install DeepSpeed ──
# DS_BUILD_CPU_ADAM=1 compiles the fused CPU Adam kernel (optional but faster)
!DS_BUILD_CPU_ADAM=1 pip install deepspeed -q

import deepspeed, torch
print(f"DeepSpeed : {deepspeed.__version__}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")

## ZeRO Memory Math

For mixed-precision training with Adam optimizer, each parameter requires:

| Component | Dtype | Bytes/param |
|-----------|-------|-------------|
| FP16 model copy | fp16 | 2 |
| FP32 master copy | fp32 | 4 |
| FP16 gradients | fp16 | 2 |
| Adam momentum | fp32 | 4 |
| Adam variance | fp32 | 4 |
| **Total** | | **16** |

So a 7B-parameter model requires **7B × 16 bytes = 112 GB** for full training — that's
across all GPUs in DDP, but each GPU holds all 112 GB. ZeRO-3 partitions it:
with 8 GPUs each holds only 14 GB.

In [ ]:
# ── ZeRO memory math ──
def zero_memory_per_gpu(num_params, num_gpus, stage):
    """Memory per GPU in GB for mixed-precision Adam training."""
    # 16 bytes per param total: 2 (fp16 model) + 4 (fp32 copy) + 2 (grads) + 4+4 (Adam)
    P = num_params
    N = num_gpus
    GB = 1e9
    if stage == 0:  # DDP baseline
        return P * 16 / GB
    elif stage == 1:  # shard optimizer states
        return P * (6 + 2 + 8/N) / GB
    elif stage == 2:  # + shard gradients
        return P * (6 + 10/N) / GB
    elif stage == 3:  # + shard parameters
        return P * 16 / N / GB

params = 7e9  # 7B model
print(f"{'N GPUs':>8} {'DDP':>10} {'ZeRO-1':>10} {'ZeRO-2':>10} {'ZeRO-3':>10}")
print('-' * 52)
for n in [1, 2, 4, 8]:
    vals = [zero_memory_per_gpu(params, n, s) for s in [0, 1, 2, 3]]
    print(f"{n:>8} " + " ".join(f"{v:>9.1f}G" for v in vals))

print("\nNote: ZeRO-3 on 8 GPUs: 14 GB/GPU vs 112 GB/GPU baseline — 8x reduction")

In [ ]:
# ── Init distributed + DeepSpeed ZeRO-1 training loop ──
# CRITICAL: torch.distributed.init_process_group() MUST be called BEFORE deepspeed.initialize()
# env vars must be set BEFORE calling init_process_group

import os
import torch
import torch.nn as nn
import torch.distributed as dist

# ── Set env vars first ──
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29500"
os.environ["RANK"]        = "0"
os.environ["LOCAL_RANK"]  = "0"
os.environ["WORLD_SIZE"]  = "1"

# ── Init process group ──
if not dist.is_initialized():
    dist.init_process_group(backend="nccl", rank=0, world_size=1)
torch.cuda.set_device(0)
print(f"Distributed initialized: rank={dist.get_rank()}, world_size={dist.get_world_size()}")

# ── Model ──
class SmallTransformer(nn.Module):
    def __init__(self, vocab=1000, d=256, heads=4, layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        enc = nn.TransformerEncoderLayer(d, heads, d*4, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, layers)
        self.head = nn.Linear(d, vocab)
    def forward(self, x):
        return self.head(self.transformer(self.embed(x)))

model = SmallTransformer()
n = sum(p.numel() for p in model.parameters())
print(f"Model params: {n:,} ({n/1e6:.1f}M)")

# ── DeepSpeed ZeRO-1 config ──
ds_config = {
    "train_batch_size"              : 16,
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps"   : 4,
    "fp16"                          : {"enabled": True},
    "zero_optimization"             : {"stage": 1},
    "optimizer": {
        "type": "AdamW",
        "params": {"lr": 1e-4, "weight_decay": 0.01}
    }
}

# ── CRITICAL: model_parameters kwarg is required ──
model_engine, optimizer, _, _ = deepspeed.initialize(
    model            = model,
    config           = ds_config,
    model_parameters = model.parameters(),   # required
)
print(f"DeepSpeed engine initialized on device: {model_engine.device}")

# ── Training loop ──
print("\nRunning 10 training steps ...")
model_engine.train()
for step in range(10):
    batch  = torch.randint(0, 1000, (4, 32)).to(model_engine.device)
    labels = torch.randint(0, 1000, (4, 32)).to(model_engine.device)

    logits = model_engine(batch)               # (4, 32, vocab)
    loss   = nn.functional.cross_entropy(
        logits.view(-1, 1000), labels.view(-1)
    )

    model_engine.backward(loss)                # DeepSpeed backward
    model_engine.step()                        # DeepSpeed optimizer step

    if step % 5 == 0:
        print(f"  step {step:3d}  loss={loss.item():.4f}")

print("Training complete!")

## ZeRO Stage Configurations (Reference)

```python
# ZeRO-2: also shards gradients
ds_config_zero2 = {
    "train_batch_size": 32,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "overlap_comm": True,
        "contiguous_gradients": True,
    },
    "optimizer": {"type": "AdamW", "params": {"lr": 1e-4}}
}

# ZeRO-3: also shards parameters (needed for 10B+ models)
ds_config_zero3 = {
    "train_batch_size": 32,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "stage3_gather_16bit_weights_on_model_save": True,
    },
    "optimizer": {"type": "AdamW", "params": {"lr": 1e-4}}
}
```

## When to Use Each Stage

| Scenario | Stage |
|----------|-------|
| Model fits with optimizer states | ZeRO-1 (minimal overhead) |
| Need gradient memory too | ZeRO-2 (most common) |
| 10B+ parameters | ZeRO-3 (linear scaling) |
| Still OOM on ZeRO-3 | ZeRO-Infinity (CPU/NVMe offload) |

## ✅ Summary

- Explained the memory redundancy problem in data-parallel training
- Showed the ZeRO math: ZeRO-3 with 8 GPUs → 8x memory reduction
- Fixed the three critical init bugs: env vars → `init_process_group` → `deepspeed.initialize` with `model_parameters`
- Ran a complete ZeRO-1 training loop on a single GPU
- Provided reference configs for ZeRO-2 and ZeRO-3